In [ ]:
import numpy as np

class GridWorldMDP:
    def __init__(self, size, goal, trap):
        self.size = size
        self.goal = goal
        self.trap = trap
        self.state_space = [(i, j) for i in range(size) for j in range(size)]
        self.action_space = ['UP', 'DOWN', 'LEFT', 'RIGHT']
    def get_next_state(self, state, action):
        i, j = state
        if action == 'UP': i -= 1
        elif action == 'DOWN': i += 1
        elif action == 'LEFT': j -= 1
        elif action == 'RIGHT': j += 1
        i = max(0, min(i, self.size - 1))
        j = max(0, min(j, self.size - 1))
        return (i, j)
    def get_reward(self, state):
        if state == self.goal:
            return 0
        elif state == self.trap:
            return -10
        return -1
def policy_iteration(mdp, gamma=0.9):
    V = {s: 0 for s in mdp.state_space}
    policy = {s: np.random.choice(mdp.action_space) for s in mdp.state_space if s != mdp.goal and s != mdp.trap}

    while True:
        # Policy Evaluation
        while True:
            delta = 0
            for s in mdp.state_space:
                if s == mdp.goal or s == mdp.trap:
                    continue
                v = V[s]
                a = policy[s]
                next_s = mdp.get_next_state(s, a)
                reward = mdp.get_reward(next_s)
                V[s] = reward + gamma * V[next_s]
                delta = max(delta, abs(v - V[s]))
            if delta < 0.01:
                break
        # Policy Improvement
        stable = True
        for s in mdp.state_space:
            if s == mdp.goal or s == mdp.trap:
                continue
            old_action = policy[s]
            best_action = None
            best_value = float('-inf')
            for a in mdp.action_space:
                next_s = mdp.get_next_state(s, a)
                reward = mdp.get_reward(next_s)
                val = reward + gamma * V[next_s]
                if val > best_value:
                    best_value = val
                    best_action = a
            policy[s] = best_action
            if old_action != best_action:
                stable = False
        if stable:
            break
    return policy, V

In [2]:
# Example
mdp = GridWorldMDP(3, (2, 2), (1, 1))
policy, values = policy_iteration(mdp)
print("\nPolicy Iteration Output:")
for s in sorted(policy):
    print(s, "->", policy[s], ", Value:", round(values[s], 3))


Policy Iteration Output:
(0, 0) -> DOWN , Value: -2.71
(0, 1) -> RIGHT , Value: -1.9
(0, 2) -> DOWN , Value: -1.0
(1, 0) -> DOWN , Value: -1.9
(1, 2) -> DOWN , Value: 0.0
(2, 0) -> RIGHT , Value: -1.0
(2, 1) -> RIGHT , Value: 0.0
